# 03 - GDELT Incremental Data Refresh

**Purpose: Scheduled incremental updates of GDELT data from US to US-CENTRAL1 region**

This notebook performs incremental data refreshes of GDELT tables with automatic deduplication:

## Workflow Overview

### Initial Load (First Run)
- Copies the last 24 hours of GDELT data from the public dataset
- Creates destination tables in US-CENTRAL1 region
- Records sync timestamp for future incremental updates

### Incremental Updates (Subsequent Runs)
- Fetches only new data since last sync (with 2-minute overlap for late arrivals)
- Merges new rows into destination tables (deduplicates by unique keys)
- Updates sync metadata timestamp
- Calls `sp_update_person_table` and `sp_update_person_cooccurrence_table` stored procedures

## Key Features

- **Cross-Region Transfer**: Handles data movement from US to US-CENTRAL1 efficiently
- **Smart Deduplication**: Uses table-specific unique keys (GKGRECORDID, GLOBALEVENTID, etc.)
- **State Tracking**: Maintains sync metadata to avoid reprocessing data
- **Overlap Buffer**: 2-minute lookback ensures late-arriving data is captured
- **Multi-Table Support**: Processes all tables in BIGQUERY_TABLES configuration

## Designed For

This notebook is designed to run every 15 minutes via a scheduler (see final cell for scheduling options).

## Prerequisites

1. Run **`01_gdelt_data_prep.ipynb` first** — it creates the partitioned tables, person-related stored procedures, and the initial dataset this notebook relies on.
2. Configure **`config.py`** (or set **`GDELT_GRAPH_DEMO_GCP_PROJECT`**; see `config.py`). **`02_gdelt_graph_create.ipynb` is not required** for refresh mechanics; run it only if you need an up-to-date **BigQuery property graph** after new data lands (re-run or recreate the graph after significant loads).
3. Ensure the BigQuery API is enabled and credentials are configured.


## 🔧 Configuration and Imports

Load configuration from `config.py` and import required libraries. Override the default GCP project with the **`GDELT_GRAPH_DEMO_GCP_PROJECT`** environment variable if needed (see `config.py`; notebooks do not take the project ID from `GOOGLE_CLOUD_PROJECT`).

In [ ]:
# Import configuration from config.py
from config import (
    GCP_PROJECT_ID,
    PROJECT_REGION,
    BIGQUERY_DATASET,
    BIGQUERY_TABLES,
    GDELT_PROJECT_ID,
    GDELT_DATASET,
    GDELT_REGION,
    print_config
)

# Import required libraries
import time
from datetime import datetime, timedelta, timezone
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
from google.auth import default

# Display configuration
print_config()

# Table-specific metadata for unique keys and timestamp columns
TABLE_METADATA = {
    "gkg_partitioned": {
        "unique_key": ["GKGRECORDID"],
        "timestamp_col": "DATE",
    },
    "events_partitioned": {
        "unique_key": ["GLOBALEVENTID"],
        "timestamp_col": "DATEADDED",
    },
    "eventmentions_partitioned": {
        "unique_key": ["GLOBALEVENTID", "MentionIdentifier", "MentionTimeDate"],
        "timestamp_col": "MentionTimeDate",
    },
}

print("\n✅ Configuration and table metadata loaded successfully!")
print(f"📊 Tables configured for incremental refresh: {list(TABLE_METADATA.keys())}")

## 🛠️ Helper Functions

Core utility functions for dataset management, state tracking, and time filtering.

In [ ]:
def ensure_datasets(client):
    """
    Ensure required datasets exist in both US-CENTRAL1 and US regions.
    Creates them if they don't exist.
    """
    print("🔍 Checking required datasets...")

    try:
        next(iter(client.list_datasets(max_results=1)), None)
    except NotFound as e:
        raise RuntimeError(
            f"BigQuery cannot access project {GCP_PROJECT_ID!r} (NotFound). "
            "Confirm the project exists, you have BigQuery access, and "
            "GDELT_GRAPH_DEMO_GCP_PROJECT / notebooks/config.py matches your GCP project."
        ) from e

    # Check/create main dataset in US-CENTRAL1
    dataset_ref = client.dataset(BIGQUERY_DATASET)
    try:
        client.get_dataset(dataset_ref)
        print(f"✅ Dataset '{BIGQUERY_DATASET}' already exists in US-CENTRAL1")
    except NotFound:
        print(f"📝 Creating dataset '{BIGQUERY_DATASET}' in US-CENTRAL1...")
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US-CENTRAL1"
        dataset.description = "GDELT data for graph analysis"
        client.create_dataset(dataset, timeout=30, exists_ok=True)
        print(f"✅ Dataset '{BIGQUERY_DATASET}' created successfully")

    # Check/create US region dataset for temporary tables
    us_dataset_name = f"{BIGQUERY_DATASET}_us"
    us_dataset_ref = bigquery.DatasetReference(GCP_PROJECT_ID, us_dataset_name)
    try:
        client.get_dataset(us_dataset_ref)
        print(f"✅ Dataset '{us_dataset_name}' already exists in US region")
    except NotFound:
        print(f"📝 Creating dataset '{us_dataset_name}' in US region...")
        us_dataset = bigquery.Dataset(us_dataset_ref)
        us_dataset.location = "US"
        us_dataset.description = "GDELT temporary staging tables (US region)"
        client.create_dataset(us_dataset, timeout=30, exists_ok=True)
        print(f"✅ Dataset '{us_dataset_name}' created in US region")

    print("✅ All required datasets are ready\n")


def get_last_sync_time(client, table_name):
    """
    Retrieve the last successful sync timestamp for a table.
    Returns None if no prior sync exists (triggers initial 24-hour load).
    """
    metadata_table = f"{GCP_PROJECT_ID}.{BIGQUERY_DATASET}._sync_metadata"
    
    try:
        query = f"""
        SELECT last_sync_time
        FROM `{metadata_table}`
        WHERE table_name = '{table_name}'
        ORDER BY last_sync_time DESC
        LIMIT 1
        """
        
        result = client.query(query, location="US-CENTRAL1").result()
        rows = list(result)
        
        if rows:
            return rows[0].last_sync_time
        else:
            return None
            
    except Exception as e:
        if "notFound" in str(e) or "404" in str(e):
            # Metadata table doesn't exist yet, create it
            print(f"📝 Creating sync metadata table...")
            create_query = f"""
            CREATE TABLE IF NOT EXISTS `{metadata_table}` (
                table_name STRING NOT NULL,
                last_sync_time TIMESTAMP NOT NULL,
                rows_processed INT64,
                sync_mode STRING,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
            )
            """
            client.query(create_query, location="US-CENTRAL1").result()
            print(f"✅ Sync metadata table created")
            return None
        else:
            print(f"⚠️  Error checking sync metadata: {e}")
            return None


def update_sync_time(client, table_name, sync_time, rows_processed, sync_mode):
    """
    Record the latest sync timestamp for a table in the metadata table.
    Uses MERGE to make this operation idempotent (safe to run multiple times).
    """
    metadata_table = f"{GCP_PROJECT_ID}.{BIGQUERY_DATASET}._sync_metadata"
    
    # Use MERGE instead of INSERT to make idempotent
    # This prevents duplicate metadata entries if function runs multiple times
    merge_query = f"""
    MERGE `{metadata_table}` AS target
    USING (
        SELECT 
            '{table_name}' as table_name,
            TIMESTAMP('{sync_time.strftime('%Y-%m-%d %H:%M:%S')}') as last_sync_time,
            {rows_processed} as rows_processed,
            '{sync_mode}' as sync_mode,
            CURRENT_TIMESTAMP() as created_at
    ) AS source
    ON target.table_name = source.table_name 
       AND TIMESTAMP_DIFF(target.last_sync_time, source.last_sync_time, SECOND) BETWEEN -60 AND 60
    WHEN NOT MATCHED THEN
        INSERT (table_name, last_sync_time, rows_processed, sync_mode, created_at)
        VALUES (source.table_name, source.last_sync_time, source.rows_processed, source.sync_mode, source.created_at)
    """
    
    try:
        client.query(merge_query, location="US-CENTRAL1").result()
        print(f"✅ Sync metadata updated for {table_name}")
    except Exception as e:
        print(f"⚠️  Error updating sync metadata: {e}")


def build_time_filter(timestamp_col, start_time, end_time):
    """
    Build a WHERE clause to filter INT64 timestamp columns (format: YYYYMMDDHHMMSS).
    
    Args:
        timestamp_col: Name of the timestamp column
        start_time: datetime object for start of range
        end_time: datetime object for end of range
    
    Returns:
        SQL WHERE clause string
    """
    start_int = int(start_time.strftime('%Y%m%d%H%M%S'))
    end_int = int(end_time.strftime('%Y%m%d%H%M%S'))
    
    return f"{timestamp_col} >= {start_int} AND {timestamp_col} <= {end_int}"


def datetime_to_int64_timestamp(dt):
    """Convert datetime to INT64 format YYYYMMDDHHMMSS."""
    return int(dt.strftime('%Y%m%d%H%M%S'))


print("✅ Helper functions loaded successfully!")

## 🔄 Core Function: Incremental Table Refresh

Main function that handles both initial loads and incremental updates for a single table.

### Edge Cases Handled

1. **NULL Unique Keys**: Rows with NULL values in unique key columns are filtered out before MERGE (cannot reliably deduplicate)
2. **NULL Timestamps**: Rows with NULL timestamp values are excluded from queries
3. **NULL Join Handling**: MERGE join uses COALESCE to treat NULL as a distinct value for comparison
4. **Zero New Rows**: When no new data is available, still updates sync timestamp to avoid reprocessing
5. **First Load**: When destination table doesn't exist, uses simple COPY instead of MERGE
6. **Cross-Region**: Properly handles data transfer from US to US-CENTRAL1 via staging tables
7. **Late Arrivals**: 2-minute overlap window ensures late-arriving data is captured in incremental runs
8. **Explicit Columns**: MERGE uses explicit column lists (BigQuery doesn't support INSERT * in MERGE statements)


In [ ]:
def incremental_refresh_table(client, table_name):
    """
    Perform incremental refresh for a single GDELT table.
    
    Process:
    1. Check last sync time (if none, do 24-hour initial load)
    2. Query GDELT data with time filter
    3. Write to temp table in US region
    4. Copy to staging table in US-CENTRAL1
    5. MERGE into destination (dedup by unique keys)
    6. Clean up temp/staging tables
    7. Update sync metadata
    
    Args:
        client: BigQuery client
        table_name: Name of the table to refresh
    
    Returns:
        dict with status and metrics
    """
    print(f"\n{'='*80}")
    print(f"🔄 Processing table: {table_name}")
    print(f"{'='*80}")
    
    try:
        # Get table metadata
        if table_name not in TABLE_METADATA:
            raise ValueError(f"Table {table_name} not found in TABLE_METADATA configuration")
        
        metadata = TABLE_METADATA[table_name]
        unique_keys = metadata["unique_key"]
        timestamp_col = metadata["timestamp_col"]
        
        # Determine sync mode and time range
        last_sync = get_last_sync_time(client, table_name)
        current_time = datetime.now(timezone.utc)
        
        if last_sync is None:
            # Initial load: last 24 hours
            sync_mode = "initial"
            start_time = current_time - timedelta(hours=24)
            print(f"📥 MODE: Initial load (last 24 hours)")
        else:
            # Incremental: since last sync with 2-minute overlap
            sync_mode = "incremental"
            start_time = last_sync - timedelta(minutes=2)
            print(f"📥 MODE: Incremental update")
            print(f"   Last sync: {last_sync}")
        
        print(f"⏰ Time range: {start_time} to {current_time}")
        
        # Build time filter
        time_filter = build_time_filter(timestamp_col, start_time, current_time)
        
        # Step 1: Query GDELT data and write to temp table in US region
        us_dataset_name = f"{BIGQUERY_DATASET}_us"
        temp_table_ref = client.dataset(us_dataset_name).table(f"staging_{table_name}")
        gdelt_table = f"{GDELT_PROJECT_ID}.{GDELT_DATASET}.{table_name}"
        
        print(f"\n📊 Step 1: Querying GDELT data...")
        print(f"   Source: {gdelt_table}")
        print(f"   Filter: {time_filter}")
        
        job_config = bigquery.QueryJobConfig()
        job_config.destination = temp_table_ref
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        job_config.create_disposition = bigquery.CreateDisposition.CREATE_IF_NEEDED
        
        partition_start = start_time.strftime('%Y-%m-%d')
        partition_end = current_time.strftime('%Y-%m-%d')

        query = f"""
        SELECT *
        FROM `{gdelt_table}`
        WHERE _PARTITIONDATE BETWEEN '{partition_start}' AND '{partition_end}'
          AND {time_filter}
          AND {timestamp_col} IS NOT NULL
        """
        
        query_job = client.query(query, job_config=job_config, location="US")
        query_job.result()
        
        # Get row count from temp table
        temp_table = client.get_table(temp_table_ref)
        temp_rows = temp_table.num_rows
        print(f"✅ Query complete: {temp_rows:,} rows fetched")
        
        if temp_rows == 0:
            print(f"ℹ️  No new data to process for {table_name}")
            # Still update sync time to avoid reprocessing
            update_sync_time(client, table_name, current_time, 0, sync_mode)
            return {"status": "success", "mode": sync_mode, "rows_processed": 0, "rows_added": 0}
        
        # Step 2: Copy to staging table in US-CENTRAL1
        print(f"\n🔄 Step 2: Copying to US-CENTRAL1 staging...")
        staging_table_ref = client.dataset(BIGQUERY_DATASET).table(f"staging_{table_name}")
        
        copy_job_config = bigquery.CopyJobConfig()
        copy_job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        copy_job_config.create_disposition = bigquery.CreateDisposition.CREATE_IF_NEEDED
        
        copy_job = client.copy_table(temp_table_ref, staging_table_ref, job_config=copy_job_config, location="US")
        copy_job.result()
        print(f"✅ Cross-region copy complete")
        
        # Step 3: MERGE into destination table
        print(f"\n🔀 Step 3: Merging into destination table...")
        dest_table = f"{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.{table_name}"
        staging_table = f"{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.staging_{table_name}"
        
        # Build join condition for unique keys with NULL handling
        # Using COALESCE to handle NULL values in join (treat NULL as matching NULL)
        join_conditions = " AND ".join([
            f"COALESCE(CAST(target.{key} AS STRING), 'NULL') = COALESCE(CAST(source.{key} AS STRING), 'NULL')"
            for key in unique_keys
        ])
        
        # Check if destination table exists
        try:
            dest_table_obj = client.get_table(dest_table)
            table_exists = True
        except Exception:
            table_exists = False
            print(f"📝 Destination table doesn't exist, will be created during merge")
        
        if table_exists:
            # Get column list from staging table for explicit INSERT
            staging_table_obj = client.get_table(staging_table_ref)
            columns = [field.name for field in staging_table_obj.schema]
            columns_list = ", ".join(columns)
            source_columns_list = ", ".join([f"source.{col}" for col in columns])
            
            # Filter out rows where unique keys are NULL (can't deduplicate properly)
            null_filters = " AND ".join([f"{key} IS NOT NULL" for key in unique_keys])
            
            # Use MERGE with explicit column list (BigQuery doesn't support INSERT * in MERGE)
            merge_query = f"""
            MERGE `{dest_table}` AS target
            USING (
              SELECT * FROM `{staging_table}`
              WHERE {null_filters}
            ) AS source
            ON {join_conditions}
            WHEN NOT MATCHED THEN
              INSERT ({columns_list})
              VALUES ({source_columns_list})
            """
            
            merge_job = client.query(merge_query, location="US-CENTRAL1")
            merge_result = merge_job.result()
            
            # Get merge stats
            rows_added = merge_job.num_dml_affected_rows if merge_job.num_dml_affected_rows else 0
            print(f"✅ Merge complete: {rows_added:,} new rows inserted")
        else:
            # First time: just copy the staging table to destination
            print(f"📝 First load: copying staging to destination...")
            first_copy_job = client.copy_table(
                staging_table_ref,
                dest_table,
                location="US-CENTRAL1"
            )
            first_copy_job.result()
            rows_added = temp_rows
            print(f"✅ Initial load complete: {rows_added:,} rows inserted")
        
        # Step 4: Cleanup temp and staging tables
        print(f"\n🧹 Step 4: Cleaning up temporary tables...")
        try:
            client.delete_table(temp_table_ref)
            print(f"✅ Deleted temp table in US region")
        except Exception as e:
            print(f"⚠️  Could not delete temp table: {e}")
        
        try:
            client.delete_table(staging_table_ref)
            print(f"✅ Deleted staging table in US-CENTRAL1")
        except Exception as e:
            print(f"⚠️  Could not delete staging table: {e}")
        
        # Step 5: Update sync metadata
        print(f"\n📝 Step 5: Updating sync metadata...")
        update_sync_time(client, table_name, current_time, rows_added, sync_mode)
        
        # Final summary
        print(f"\n✅ Successfully processed {table_name}")
        print(f"   Mode: {sync_mode}")
        print(f"   Rows fetched: {temp_rows:,}")
        print(f"   Rows added: {rows_added:,}")
        
        return {
            "status": "success",
            "mode": sync_mode,
            "rows_processed": temp_rows,
            "rows_added": rows_added
        }
        
    except Exception as e:
        print(f"\n❌ Error processing {table_name}: {e}")
        import traceback
        traceback.print_exc()
        return {
            "status": "failed",
            "error": str(e)
        }


print("✅ Core refresh function loaded successfully!")

## 🚀 Orchestration Function: Refresh All Tables

Coordinates the refresh process across all configured GDELT tables.

In [ ]:
def run_refresh_all_tables():
    """
    Run incremental refresh for all tables in BIGQUERY_TABLES configuration.
    
    Returns:
        dict with results for each table
    """
    print("🚀 Starting incremental refresh for all GDELT tables...")
    print(f"📅 Execution time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(f"📊 Tables to refresh: {BIGQUERY_TABLES}")
    print("=" * 80)
    
    # Create BigQuery client
    client = bigquery.Client(project=GCP_PROJECT_ID)
    print(f"✅ Connected to BigQuery project: {GCP_PROJECT_ID}\n")
    
    # Ensure datasets exist
    ensure_datasets(client)
    
    # Process each table
    results = {}
    start_time = time.time()
    
    for table_name in BIGQUERY_TABLES:
        if table_name not in TABLE_METADATA:
            print(f"⚠️  Skipping {table_name}: not found in TABLE_METADATA")
            results[table_name] = {"status": "skipped", "reason": "not_configured"}
            continue
        
        result = incremental_refresh_table(client, table_name)
        results[table_name] = result
    
    # Print final summary
    elapsed_time = time.time() - start_time
    
    print(f"\n\n{'='*80}")
    print("📊 FINAL SUMMARY")
    print(f"{'='*80}")
    
    success_count = sum(1 for r in results.values() if r.get("status") == "success")
    failed_count = sum(1 for r in results.values() if r.get("status") == "failed")
    total_rows_added = sum(r.get("rows_added", 0) for r in results.values() if r.get("status") == "success")
    
    print(f"✅ Successfully processed: {success_count}/{len(BIGQUERY_TABLES)} tables")
    print(f"❌ Failed: {failed_count}/{len(BIGQUERY_TABLES)} tables")
    print(f"📊 Total rows added across all tables: {total_rows_added:,}")
    print(f"⏱️  Total execution time: {elapsed_time:.2f} seconds")
    
    print(f"\nDetailed results:")
    for table_name, result in results.items():
        status_emoji = "✅" if result.get("status") == "success" else "❌"
        print(f"\n  {status_emoji} {table_name}:")
        print(f"      Status: {result.get('status')}")
        if "mode" in result:
            print(f"      Mode: {result['mode']}")
        if "rows_processed" in result:
            print(f"      Rows processed: {result['rows_processed']:,}")
        if "rows_added" in result:
            print(f"      Rows added: {result['rows_added']:,}")
        if "error" in result:
            print(f"      Error: {result['error']}")
    
    print(f"\n{'='*80}")
    print("🎉 Incremental refresh completed!")
    print(f"{'='*80}\n")
    
    return results


print("✅ Orchestration function loaded successfully!")

## 🔄 Idempotency & Safety Guarantees

This implementation is **fully idempotent** - it can be run multiple times safely:

### ✅ What Makes It Idempotent

1. **Data Deduplication**:
   - Uses `MERGE` with unique key matching
   - `WHEN NOT MATCHED` ensures only new rows are inserted
   - Running twice on same time window: no duplicate data

2. **Metadata Tracking**:
   - Uses `MERGE` instead of `INSERT` for metadata
   - Matches on table_name + timestamp (within 60 seconds)
   - Prevents duplicate metadata entries

3. **Staging Tables**:
   - Uses `WRITE_TRUNCATE` - overwrites each time
   - Cleanup at end removes temp tables
   - No accumulation of staging data

### 🧪 Test Scenarios

**Scenario 1**: Run twice in quick succession
```
Run 1: Processes 10,000 rows → Inserts 10,000 rows
Run 2: Processes same 10,000 rows → Inserts 0 rows (MERGE skips duplicates)
Result: Only 10,000 rows in destination ✅
```

**Scenario 2**: Scheduler triggers while manual run is in progress
```
Both runs: Query same time window
Result: MERGE ensures no duplicates ✅
```

**Scenario 3**: Function fails mid-execution
```
Partial data in staging tables → Cleaned up next run
No metadata update → Next run treats as incomplete
Result: Safe to retry ✅
```

### 🎯 Best Practices

- **Safe to re-run**: If uncertain, just run it again
- **Manual + Scheduled**: Both can run simultaneously without conflicts
- **Overlapping windows**: 2-minute overlap intentionally duplicates queries, MERGE handles it
- **Cost optimization**: MERGE only processes new rows, so re-runs are cheap


## ▶️ Execute Refresh

Run the incremental refresh process for all configured tables.

In [ ]:
# Execute the incremental refresh
results = run_refresh_all_tables()

## 👥 Post-Refresh: Update Person Tables

After each incremental update of `gkg_partitioned`, call the BigQuery stored procedures to keep
`person` and `person_cooccurrence` in sync with the latest data.

The stored procedures are created by `01_gdelt_data_prep.ipynb` and live in the `gdelt` dataset.
They accept a `TIMESTAMP` parameter: the last GKG sync time is passed so only new rows are processed.


In [ ]:
from google.cloud import bigquery as _bq

_client = _bq.Client(project=GCP_PROJECT_ID)

# Use the last gkg sync time so person updates only process the new GKG rows.
# Returns None on first run — the stored procedures handle that as a full rebuild.
_since = get_last_sync_time(_client, "gkg_partitioned")
print(f"📅 Updating person tables for data since: {_since or 'beginning (first run)'}")

if _since is not None:
    _since_param = f"TIMESTAMP('{_since.strftime('%Y-%m-%d %H:%M:%S')}')" 
else:
    _since_param = "NULL"

print(f"\n{'='*80}\n👥 Updating person table...")
try:
    _job = _client.query(
        f"CALL `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.sp_update_person_table`({_since_param})",
        location="US-CENTRAL1"
    )
    _job.result()
    print(f"✅ Person table updated: {_job.num_dml_affected_rows or 0:,} rows affected")
    person_ok = True
except Exception as e:
    print(f"❌ Error updating person table: {e}")
    person_ok = False

print(f"\n{'='*80}\n🔗 Updating person_cooccurrence table...")
try:
    _job2 = _client.query(
        f"CALL `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.sp_update_person_cooccurrence_table`({_since_param})",
        location="US-CENTRAL1"
    )
    _job2.result()
    print(f"✅ Person cooccurrence table updated")
    cooccurrence_ok = True
except Exception as e:
    print(f"❌ Error updating person_cooccurrence table: {e}")
    cooccurrence_ok = False

print(f"\n{'='*80}")
print(f"Person table update:        {'success' if person_ok else 'FAILED'}")
print(f"Co-occurrence table update: {'success' if cooccurrence_ok else 'FAILED'}")


## ☁️ Cloud Function Deployment

Deploy this refresh logic as a Cloud Function with Cloud Scheduler for automatic 15-minute updates.

**Production vs notebook-generated files:** For deployments you plan to operate long term, prefer **`notebooks/cloud_function/deploy.sh`** (and the checked-in `main.py` / `requirements.txt` there). It copies shared **`notebooks/config.py`** before upload. The cells below are useful to **experiment**, regenerate stubs, or align the folder with logic you prototyped in the notebook—they are not a second source of truth alongside `deploy.sh`.

**Run the cells below to**:
1. Create Cloud Function code files
2. Deploy to Google Cloud
3. Set up Cloud Scheduler
4. Test and monitor


### Step 1: Create Cloud Function Files

This cell creates the necessary files for the Cloud Function deployment.

In [ ]:
import os
import shutil
from pathlib import Path

# Create cloud_function directory
cf_dir = Path("cloud_function")
cf_dir.mkdir(exist_ok=True)

print("📁 Creating Cloud Function files...")

# Create complete main.py with all the logic
main_py_content = '''"""GDELT Incremental Refresh Cloud Function."""
import functions_framework
from google.cloud import bigquery
from datetime import datetime, timedelta
import logging
import config

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

TABLE_METADATA = {
    "gkg_partitioned": {"unique_key": ["GKGRECORDID"], "timestamp_col": "DATE"},
    "events_partitioned": {"unique_key": ["GLOBALEVENTID"], "timestamp_col": "DATEADDED"},
    "eventmentions_partitioned": {"unique_key": ["GLOBALEVENTID", "MentionIdentifier", "MentionTimeDate"], "timestamp_col": "MentionTimeDate"},
}

def ensure_datasets(client):
    dataset_ref = client.dataset(config.BIGQUERY_DATASET)
    try:
        client.get_dataset(dataset_ref)
    except:
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US-CENTRAL1"
        client.create_dataset(dataset, timeout=30, exists_ok=True)
    us_dataset_name = f"{config.BIGQUERY_DATASET}_us"
    us_dataset_ref = bigquery.DatasetReference(config.GCP_PROJECT_ID, us_dataset_name)
    try:
        client.get_dataset(us_dataset_ref)
    except:
        us_dataset = bigquery.Dataset(us_dataset_ref)
        us_dataset.location = "US"
        client.create_dataset(us_dataset, timeout=30, exists_ok=True)

def get_last_sync_time(client, table_name):
    metadata_table = f"{config.GCP_PROJECT_ID}.{config.BIGQUERY_DATASET}._sync_metadata"
    try:
        query = f"SELECT last_sync_time FROM `{metadata_table}` WHERE table_name = '{table_name}' ORDER BY last_sync_time DESC LIMIT 1"
        result = client.query(query, location="US-CENTRAL1").result()
        rows = list(result)
        return rows[0].last_sync_time if rows else None
    except Exception as e:
        if "notFound" in str(e) or "404" in str(e):
            create_query = f"""CREATE TABLE IF NOT EXISTS `{metadata_table}` (
                table_name STRING NOT NULL, last_sync_time TIMESTAMP NOT NULL,
                rows_processed INT64, sync_mode STRING, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP())"""
            client.query(create_query, location="US-CENTRAL1").result()
            return None
        return None

def update_sync_time(client, table_name, sync_time, rows_processed, sync_mode):
    metadata_table = f"{config.GCP_PROJECT_ID}.{config.BIGQUERY_DATASET}._sync_metadata"
    merge_query = f"""MERGE `{metadata_table}` AS target USING (
        SELECT '{table_name}' as table_name, TIMESTAMP('{sync_time.strftime('%Y-%m-%d %H:%M:%S')}') as last_sync_time,
        {rows_processed} as rows_processed, '{sync_mode}' as sync_mode, CURRENT_TIMESTAMP() as created_at
    ) AS source ON target.table_name = source.table_name 
       AND TIMESTAMP_DIFF(target.last_sync_time, source.last_sync_time, SECOND) BETWEEN -60 AND 60
    WHEN NOT MATCHED THEN INSERT (table_name, last_sync_time, rows_processed, sync_mode, created_at)
        VALUES (source.table_name, source.last_sync_time, source.rows_processed, source.sync_mode, source.created_at)"""
    try:
        client.query(merge_query, location="US-CENTRAL1").result()
    except Exception as e:
        logger.warning(f"Error updating metadata: {e}")

def build_time_filter(timestamp_col, start_time, end_time):
    start_int = int(start_time.strftime('%Y%m%d%H%M%S'))
    end_int = int(end_time.strftime('%Y%m%d%H%M%S'))
    return f"{timestamp_col} >= {start_int} AND {timestamp_col} <= {end_int}"

def incremental_refresh_table(client, table_name):
    logger.info(f"Processing {table_name}")
    try:
        metadata = TABLE_METADATA[table_name]
        unique_keys = metadata["unique_key"]
        timestamp_col = metadata["timestamp_col"]
        last_sync = get_last_sync_time(client, table_name)
        current_time = datetime.utcnow()
        if last_sync is None:
            sync_mode = "initial"
            start_time = current_time - timedelta(hours=24)
        else:
            sync_mode = "incremental"
            start_time = last_sync - timedelta(minutes=2)
        time_filter = build_time_filter(timestamp_col, start_time, current_time)
        us_dataset_name = f"{config.BIGQUERY_DATASET}_us"
        temp_table_ref = client.dataset(us_dataset_name).table(f"staging_{table_name}")
        gdelt_table = f"{config.GDELT_PROJECT_ID}.{config.GDELT_DATASET}.{table_name}"
        job_config = bigquery.QueryJobConfig()
        job_config.destination = temp_table_ref
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        job_config.create_disposition = bigquery.CreateDisposition.CREATE_IF_NEEDED
        partition_start = start_time.strftime('%Y-%m-%d')
        partition_end = current_time.strftime('%Y-%m-%d')
        query = f"""
        SELECT *
        FROM `{gdelt_table}`
        WHERE _PARTITIONDATE BETWEEN '{partition_start}' AND '{partition_end}'
          AND {time_filter}
          AND {timestamp_col} IS NOT NULL
        """
        client.query(query, job_config=job_config, location="US").result()
        temp_table = client.get_table(temp_table_ref)
        temp_rows = temp_table.num_rows
        if temp_rows == 0:
            update_sync_time(client, table_name, current_time, 0, sync_mode)
            return {"status": "success", "mode": sync_mode, "rows_processed": 0, "rows_added": 0}
        staging_table_ref = client.dataset(config.BIGQUERY_DATASET).table(f"staging_{table_name}")
        copy_job_config = bigquery.CopyJobConfig()
        copy_job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        copy_job_config.create_disposition = bigquery.CreateDisposition.CREATE_IF_NEEDED
        client.copy_table(temp_table_ref, staging_table_ref, job_config=copy_job_config, location="US").result()
        dest_table = f"{config.GCP_PROJECT_ID}.{config.BIGQUERY_DATASET}.{table_name}"
        staging_table = f"{config.GCP_PROJECT_ID}.{config.BIGQUERY_DATASET}.staging_{table_name}"
        join_conditions = " AND ".join([f"COALESCE(CAST(target.{key} AS STRING), 'NULL') = COALESCE(CAST(source.{key} AS STRING), 'NULL')" for key in unique_keys])
        try:
            client.get_table(dest_table)
            table_exists = True
        except:
            table_exists = False
        if table_exists:
            staging_table_obj = client.get_table(staging_table_ref)
            columns = [field.name for field in staging_table_obj.schema]
            columns_list = ", ".join(columns)
            source_columns_list = ", ".join([f"source.{col}" for col in columns])
            null_filters = " AND ".join([f"{key} IS NOT NULL" for key in unique_keys])
            merge_query = f"""MERGE `{dest_table}` AS target USING (
                SELECT * FROM `{staging_table}` WHERE {null_filters}) AS source
                ON {join_conditions} WHEN NOT MATCHED THEN INSERT ({columns_list}) VALUES ({source_columns_list})"""
            merge_job = client.query(merge_query, location="US-CENTRAL1")
            merge_job.result()
            rows_added = merge_job.num_dml_affected_rows if merge_job.num_dml_affected_rows else 0
        else:
            client.copy_table(staging_table_ref, dest_table, location="US-CENTRAL1").result()
            rows_added = temp_rows
        try:
            client.delete_table(temp_table_ref)
            client.delete_table(staging_table_ref)
        except:
            pass
        update_sync_time(client, table_name, current_time, rows_added, sync_mode)
        logger.info(f"Completed {table_name}: {rows_added} rows")
        return {"status": "success", "mode": sync_mode, "rows_processed": temp_rows, "rows_added": rows_added}
    except Exception as e:
        logger.error(f"Error: {e}")
        return {"status": "failed", "error": str(e)}

def update_person_tables(client):
    """Call stored procedures to update person and person_cooccurrence tables."""
    since = get_last_sync_time(client, "gkg_partitioned")
    if since is not None:
        since_param = f"TIMESTAMP('{since.strftime('%Y-%m-%d %H:%M:%S')}')"
    else:
        since_param = "NULL"

    person_result = {"status": "success"}
    cooccurrence_result = {"status": "success"}

    try:
        logger.info("Updating person table")
        client.query(
            f"CALL `{config.GCP_PROJECT_ID}.{config.BIGQUERY_DATASET}.sp_update_person_table`({since_param})",
            location="US-CENTRAL1"
        ).result()
        logger.info("Person table updated")
    except Exception as e:
        logger.error(f"Error updating person table: {e}")
        person_result = {"status": "failed", "error": str(e)}

    try:
        logger.info("Updating person_cooccurrence table")
        client.query(
            f"CALL `{config.GCP_PROJECT_ID}.{config.BIGQUERY_DATASET}.sp_update_person_cooccurrence_table`({since_param})",
            location="US-CENTRAL1"
        ).result()
        logger.info("Person cooccurrence table updated")
    except Exception as e:
        logger.error(f"Error updating person_cooccurrence table: {e}")
        cooccurrence_result = {"status": "failed", "error": str(e)}

    return {"person": person_result, "person_cooccurrence": cooccurrence_result}

@functions_framework.http
def gdelt_refresh(request):
    try:
        logger.info("Starting refresh")
        client = bigquery.Client(project=config.GCP_PROJECT_ID)
        ensure_datasets(client)
        results = {}
        for table_name in config.BIGQUERY_TABLES:
            if table_name in TABLE_METADATA:
                results[table_name] = incremental_refresh_table(client, table_name)
        success_count = sum(1 for r in results.values() if r.get("status") == "success")
        total_rows = sum(r.get("rows_added", 0) for r in results.values())
        logger.info(f"Complete: {success_count} tables, {total_rows} rows")
        person_results = update_person_tables(client)
        return {"status": "completed", "timestamp": datetime.utcnow().isoformat(),
                "summary": {"successful": success_count, "total_rows_added": total_rows},
                "details": results, "person_updates": person_results}, 200
    except Exception as e:
        logger.error(f"Error: {e}")
        return {"status": "error", "message": str(e)}, 500

'''

with open(cf_dir / "main.py", "w") as f:
    f.write(main_py_content)
print("✅ Created main.py")

# Shared config (same file as the notebooks; deploy.sh also copies ../config.py before gcloud deploy)
shared_config = Path("config.py")
if not shared_config.is_file():
    raise FileNotFoundError(
        "Missing notebooks/config.py — run this notebook with working directory set to notebooks/."
    )
shutil.copy(shared_config, cf_dir / "config.py")
print("✅ Copied config.py from notebooks/")

# Requirements
with open(cf_dir / "requirements.txt", "w") as f:
    f.write("functions-framework==3.*\ngoogle-cloud-bigquery==3.*\n")
print("✅ Created requirements.txt")

print(f"\n📁 Files ready in: {cf_dir.absolute()}")
print("\n📋 Files created:")
for file in ["main.py", "config.py", "requirements.txt"]:
    filepath = cf_dir / file
    if filepath.exists():
        print(f"   ✅ {file} ({filepath.stat().st_size} bytes)")


### Step 2: Deploy Cloud Function

**Note**: Requires `gcloud` CLI installed and authenticated.

Run this cell to deploy the function to Google Cloud.

In [ ]:
import subprocess

print("🔧 Step 2a: Setting up APIs, Artifact Registry, and permissions...\n")

# 1. Enable APIs
print("1️⃣ Enabling APIs...")
apis = [
    "cloudfunctions.googleapis.com",
    "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com",
    "run.googleapis.com",
    "cloudscheduler.googleapis.com"
]
for api in apis:
    cmd = ["gcloud", "services", "enable", api]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"   ✅ {api}")

# 2. Create the gcf-artifacts repository if it doesn't exist
print(f"\n2️⃣ Creating Artifact Registry repository...")
repo_cmd = [
    "gcloud", "artifacts", "repositories", "create", "gcf-artifacts",
    "--repository-format=docker",
    f"--location={PROJECT_REGION}",
    "--description=Cloud Functions container images"
]
result = subprocess.run(repo_cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("   ✅ Repository 'gcf-artifacts' created")
elif "already exists" in result.stderr.lower() or "ALREADY_EXISTS" in result.stderr:
    print("   ✅ Repository 'gcf-artifacts' already exists")
else:
    print(f"   ⚠️  {result.stderr.strip()[:200]}")

# 3. Get project number and grant permissions
print(f"\n3️⃣ Granting Artifact Registry permissions to Cloud Build...")
proj_num_cmd = ["gcloud", "projects", "describe", GCP_PROJECT_ID, "--format=value(projectNumber)"]
proj_num_result = subprocess.run(proj_num_cmd, capture_output=True, text=True)
project_number = proj_num_result.stdout.strip()
print(f"   Project number: {project_number}")

# Grant permissions to Cloud Build service account
service_accounts = [
    f"{project_number}@cloudbuild.gserviceaccount.com",
    f"{project_number}-compute@developer.gserviceaccount.com"
]

for sa in service_accounts:
    for role in ["roles/artifactregistry.writer", "roles/artifactregistry.reader"]:
        iam_cmd = [
            "gcloud", "projects", "add-iam-policy-binding", GCP_PROJECT_ID,
            f"--member=serviceAccount:{sa}",
            f"--role={role}",
            "--condition=None",
            "--quiet"
        ]
        result = subprocess.run(iam_cmd, capture_output=True, text=True)
    print(f"   ✅ Granted Artifact Registry permissions to {sa}")

# 3b. Cloud Functions Gen2 deploy runs builds that need logs, source buckets (gcf-v2-sources-*),
# and Artifact Registry. See https://cloud.google.com/functions/docs/troubleshooting#build-service-account
print(f"\n3️⃣b Granting Cloud Build pipeline permissions (Gen2 deploy)...")
build_pipeline_roles = [
    "roles/logging.logWriter",
    "roles/storage.objectViewer",
]
for sa in service_accounts:
    for role in build_pipeline_roles:
        iam_cmd = [
            "gcloud", "projects", "add-iam-policy-binding", GCP_PROJECT_ID,
            f"--member=serviceAccount:{sa}",
            f"--role={role}",
            "--condition=None",
            "--quiet",
        ]
        result = subprocess.run(iam_cmd, capture_output=True, text=True)
        status = "✅" if result.returncode == 0 else "⚠️ "
        print(f"   {status} {role} → {sa}")

compute_sa = f"{project_number}-compute@developer.gserviceaccount.com"
for role in ["roles/cloudbuild.builds.builder"]:
    iam_cmd = [
        "gcloud", "projects", "add-iam-policy-binding", GCP_PROJECT_ID,
        f"--member=serviceAccount:{compute_sa}",
        f"--role={role}",
        "--condition=None",
        "--quiet",
    ]
    result = subprocess.run(iam_cmd, capture_output=True, text=True)
    status = "✅" if result.returncode == 0 else "⚠️ "
    print(f"   {status} {role} → {compute_sa}")

# 4. Grant BigQuery permissions to the Cloud Function's runtime service account
print(f"\n4️⃣ Granting BigQuery permissions to Cloud Function runtime service account...")
cf_sa = f"{project_number}-compute@developer.gserviceaccount.com"
for role in ["roles/bigquery.dataEditor", "roles/bigquery.jobUser"]:
    iam_cmd = [
        "gcloud", "projects", "add-iam-policy-binding", GCP_PROJECT_ID,
        f"--member=serviceAccount:{cf_sa}",
        f"--role={role}",
        "--condition=None",
        "--quiet"
    ]
    result = subprocess.run(iam_cmd, capture_output=True, text=True)
    status = "✅" if result.returncode == 0 else "⚠️ "
    print(f"   {status} {role}")
print(f"   Granted BigQuery permissions to {cf_sa}")

print("\n✅ Setup complete! Run the next cell to deploy.")

### Step 2b: Deploy Cloud Function

Now deploy the function (APIs should be enabled from previous cell).

In [ ]:
import subprocess

FUNCTION_NAME = "gdelt-incremental-refresh"
REGION = "us-central1"

print("🚀 Deploying Cloud Function...\n")

deploy_cmd = [
    "gcloud", "functions", "deploy", FUNCTION_NAME,
    "--gen2",
    "--runtime=python312",
    f"--region={REGION}",
    "--source=./cloud_function",
    "--entry-point=gdelt_refresh",
    "--trigger-http",
    "--no-allow-unauthenticated",
    "--memory=2048MB",
    "--timeout=540s"
]

print("Running deployment...\n")
result = subprocess.run(deploy_cmd, capture_output=False, text=True)

if result.returncode == 0:
    print("\n✅ Deployed successfully!")
    # Gen2 functions are Cloud Run services; without public invoker, grant the default
    # compute service account (used by Cloud Scheduler with OIDC) permission to invoke.
    proj_n = subprocess.run(
        ["gcloud", "projects", "describe", GCP_PROJECT_ID, "--format=value(projectNumber)"],
        capture_output=True,
        text=True,
    ).stdout.strip()
    invoker = f"{proj_n}-compute@developer.gserviceaccount.com"
    subprocess.run(
        [
            "gcloud", "run", "services", "add-iam-policy-binding", FUNCTION_NAME,
            f"--region={REGION}",
            f"--member=serviceAccount:{invoker}",
            "--role=roles/run.invoker",
            "--quiet",
        ],
        capture_output=False,
    )
    print(f"   Granted roles/run.invoker on the function to {invoker}")
    
    # Get URL
    url_cmd = ["gcloud", "functions", "describe", FUNCTION_NAME,
               f"--region={REGION}", "--gen2", "--format=value(serviceConfig.uri)"]
    url_result = subprocess.run(url_cmd, capture_output=True, text=True)
    
    if url_result.returncode == 0:
        function_url = url_result.stdout.strip()
        print(f"\n📍 Function URL: {function_url}")
        
        # Save for next step
        with open("cloud_function/.function_url", "w") as f:
            f.write(function_url)
        print("\n✅ URL saved for scheduler setup")
else:
    print("\n❌ Deployment failed.")
    print("\n💡 Troubleshooting:")
    print("   1. Wait 30 seconds after enabling APIs, then try again")
    print("   2. Check if you have billing enabled on your project")
    print("   3. Verify your gcloud is authenticated: gcloud auth list")
    print("   4. If the error mentions IAM policy / permitted customer, your org may")
    print("      block public invoker (allUsers). This notebook uses --no-allow-unauthenticated")
    print("      and OIDC from Scheduler — align with cloud_function/deploy.sh if unsure.")
    print("   5. Try deploying manually to see detailed errors:")
    print("      cd cloud_function && gcloud functions deploy gdelt-incremental-refresh \\")
    print("        --gen2 --runtime=python312 --region=us-central1 --source=. \\")
    print("        --entry-point=gdelt_refresh --trigger-http --no-allow-unauthenticated \\")
    print("        --memory=2048MB --timeout=540s")

### Step 3: Set Up Cloud Scheduler

Create a scheduler job to run the function every 15 minutes.

In [ ]:
SCHEDULER_JOB_NAME = "gdelt-refresh-scheduler"
SCHEDULE = "*/15 * * * *"  # Every 15 minutes

# Read function URL
with open("cloud_function/.function_url", "r") as f:
    function_url = f.read().strip()

proj_n = subprocess.run(
    ["gcloud", "projects", "describe", GCP_PROJECT_ID, "--format=value(projectNumber)"],
    capture_output=True,
    text=True,
).stdout.strip()
invoker_sa = f"{proj_n}-compute@developer.gserviceaccount.com"

print(f"⏰ Setting up scheduler: {SCHEDULE}\n")

# Idempotent: update existing job or create if missing
# OIDC: private Gen2 function requires an identity token (same pattern as cloud_function/deploy.sh).
common_args = [
    f"--location={REGION}",
    f"--schedule={SCHEDULE}",
    f"--uri={function_url}",
    "--http-method=POST",
    "--time-zone=UTC",
    f"--oidc-service-account-email={invoker_sa}",
    f"--oidc-token-audience={function_url}",
]

describe_cmd = [
    "gcloud", "scheduler", "jobs", "describe",
    SCHEDULER_JOB_NAME,
    f"--location={REGION}",
]
describe_result = subprocess.run(describe_cmd, capture_output=True, text=True)

if describe_result.returncode == 0:
    print("ℹ️ Scheduler job already exists; updating configuration...")
    result = subprocess.run(
        [
            "gcloud",
            "scheduler",
            "jobs",
            "update",
            "http",
            SCHEDULER_JOB_NAME,
            *common_args,
        ],
        capture_output=True,
        text=True,
    )
else:
    result = subprocess.run(
        [
            "gcloud",
            "scheduler",
            "jobs",
            "create",
            "http",
            SCHEDULER_JOB_NAME,
            *common_args,
        ],
        capture_output=True,
        text=True,
    )

combined = ((result.stderr or "") + (result.stdout or "")).lower()
if result.returncode == 0:
    print("✅ Scheduler configured!")
    print("\n🎉 Setup complete! Data will refresh every 15 minutes.")
elif "already exists" in combined or "already_exists" in combined:
    print("✅ Scheduler already exists — treating as success.")
    print("\n🎉 Setup complete! Data will refresh every 15 minutes.")
else:
    print(f"❌ Error: {result.stderr or result.stdout}")

### Step 4: Test & Monitor

Trigger a test run and view logs.

In [ ]:
# Trigger test (RunJob requires the scheduler job to be ENABLED, not PAUSED)
print("🧪 Triggering test run...\n")
subprocess.run(
    ["gcloud", "scheduler", "jobs", "resume", SCHEDULER_JOB_NAME, f"--location={REGION}"],
    capture_output=True,
    text=True,
)
trigger_cmd = ["gcloud", "scheduler", "jobs", "run", SCHEDULER_JOB_NAME, f"--location={REGION}"]
subprocess.run(trigger_cmd)

print("\n⏳ Function executing... Check logs in next cell (wait ~30 seconds)")

In [ ]:
# View logs
print("📋 Recent logs:\n")
logs_cmd = ["gcloud", "functions", "logs", "read", FUNCTION_NAME,
            f"--region={REGION}", "--gen2", "--limit=20"]
subprocess.run(logs_cmd)

In [ ]:
# Check sync status
print("📊 Sync status:\n")
query = f"""
SELECT table_name, last_sync_time, sync_mode, rows_processed,
       TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), last_sync_time, MINUTE) as minutes_ago
FROM `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}._sync_metadata`
ORDER BY last_sync_time DESC LIMIT 5
"""
client = bigquery.Client(project=GCP_PROJECT_ID)
df = client.query(query).to_dataframe()
print(df.to_string())

### Management Commands

In [ ]:
# Pause scheduler
!gcloud scheduler jobs pause {SCHEDULER_JOB_NAME} --location={REGION}
print("⏸️  Scheduler paused")

In [ ]:
# Resume scheduler
!gcloud scheduler jobs resume {SCHEDULER_JOB_NAME} --location={REGION}
print("▶️  Scheduler resumed")

## ⏰ Alternative Scheduling Options

This notebook is designed to run every 15 minutes. Here are several options for scheduling:

### Option 1: Google Cloud Scheduler + Cloud Functions (Recommended for Production)

**Best for**: Production deployments with monitoring and error handling

**✅ READY TO USE: Pre-built deployment in `cloud_function/` directory**

We've created a complete Cloud Function implementation for you:

```bash
cd cloud_function
./deploy.sh
```

This automated script will:
1. Deploy the Cloud Function (Python code equivalent to this notebook)
2. Set up Cloud Scheduler to run every 15 minutes
3. Configure all necessary permissions and settings

**Files included**:
- `main.py` - Cloud Function code with all refresh logic
- `requirements.txt` - Python dependencies
- `deploy.sh` - Copies `../config.py`, then deploys (same settings as the notebooks)
- `README.md` - Complete documentation

Configuration lives in **`notebooks/config.py`** (env vars like `GDELT_GRAPH_DEMO_GCP_PROJECT`, or edit defaults there). `deploy.sh` copies that file into `cloud_function/` before upload.

**Quick start**:
```bash
# 1. Update shared config (from repo root: notebooks/config.py)
#    Or export GDELT_GRAPH_DEMO_GCP_PROJECT=your-project

# 2. Deploy (requires gcloud CLI)
cd cloud_function
./deploy.sh

# 3. Test immediately
gcloud scheduler jobs run gdelt-refresh-scheduler --location=us-central1

# 4. Monitor
gcloud functions logs read gdelt-incremental-refresh --region=us-central1 --gen2 --limit=50
```

**Features**:
- ✅ Automatic retry on failure
- ✅ Structured logging to Cloud Logging
- ✅ HTTP endpoint for manual triggering
- ✅ 9-minute timeout (enough for large datasets)
- ✅ 2GB memory allocation
- ✅ Monitoring and alerting ready

**Estimated cost**: ~$15-60/month depending on data volume

See `cloud_function/README.md` for complete documentation.

### Option 2: Vertex AI Workbench Scheduled Execution

**Best for**: Integrated notebook environment

1. Open this notebook in Vertex AI Workbench
2. Click **Edit** → **Notebook Settings**
3. Enable **Scheduled Runs**
4. Set schedule: Every 15 minutes (cron: `*/15 * * * *`)
5. Configure execution parameters and save

### Option 3: Local Cron Job

**Best for**: Development and testing

Add to your crontab (`crontab -e`):
```bash
*/15 * * * * cd /path/to/notebooks && jupyter nbconvert --execute --to notebook --inplace 03_gdelt_incremental_refresh.ipynb >> /tmp/gdelt-refresh.log 2>&1
```

### Option 4: Cloud Composer (Apache Airflow)

**Best for**: Complex data pipelines with dependencies

Create an Airflow DAG:
```python
from airflow import DAG
from airflow.providers.google.cloud.operators.bigquery import BigQueryExecuteQueryOperator
from datetime import datetime, timedelta

dag = DAG(
    'gdelt_incremental_refresh',
    schedule_interval='*/15 * * * *',
    start_date=datetime(2026, 3, 1),
    catchup=False
)

# Add tasks to execute the refresh logic
```

### Monitoring and Alerts

Regardless of scheduling method, consider:

- **Cloud Monitoring**: Set up alerts for failed executions
- **Log Analysis**: Monitor sync metadata table for processing gaps
- **BigQuery Metrics**: Track query costs and performance
- **Row Count Validation**: Compare source vs destination row counts periodically

### Checking Sync Status

Query the metadata table to check sync status:
```sql
SELECT 
  table_name,
  last_sync_time,
  sync_mode,
  rows_processed,
  TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), last_sync_time, MINUTE) as minutes_since_sync
FROM `YOUR_PROJECT.gdelt._sync_metadata`
ORDER BY last_sync_time DESC
LIMIT 10;
```

### Cost Optimization Tips

1. **Adjust overlap window**: The 2-minute overlap ensures late data is captured, but can be adjusted based on your needs
2. **Monitor query costs**: Use BigQuery query logs to track processing costs
3. **Partition pruning**: Source GDELT tables in BigQuery are partitioned; time-bounded queries (like this pipeline’s filters) should prune partitions when predicates align with each table’s partition columns—avoid scanning full history unless intentional.
4. **Cleanup staging tables**: Ensure temp/staging tables are removed after each run (this notebook’s flow truncates or drops staging paths as designed).
